# Optimizer Results Inspector
Load and visualise the output of a portfolio optimisation run.

**Requires:** run `python main.py` first to generate CSVs in `data/processed/bids/`

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BIDS_DIR = Path('../data/processed/bids')

# Pick the most recent run automatically
fi_schedules = sorted(BIDS_DIR.glob('*_FI_schedule.csv'))
if not fi_schedules:
    raise FileNotFoundError(f'No FI schedule CSVs found in {BIDS_DIR}. Run main.py first.')

latest = fi_schedules[-1].stem.split('_')[0]
print(f'Loading run: {latest}')

fi_path   = BIDS_DIR / f'{latest}_FI_schedule.csv'
se2_path  = BIDS_DIR / f'{latest}_SE2_schedule.csv'
bids_path = BIDS_DIR / f'{latest}_FI_elspot_bids.csv'

In [ ]:
fi = pd.read_csv(fi_path, index_col='timestamp', parse_dates=True)
fi.index = pd.DatetimeIndex(fi.index, tz='UTC')
print(f'FI schedule: {len(fi)} hours  |  columns: {list(fi.columns)}')
fi.head(3)

In [ ]:
se2 = None
if se2_path.exists():
    se2 = pd.read_csv(se2_path, index_col='timestamp', parse_dates=True)
    se2.index = pd.DatetimeIndex(se2.index, tz='UTC')
    print(f'SE2 schedule: {len(se2)} hours  |  columns: {list(se2.columns)}')
else:
    print('No SE2 schedule found for this run.')

## Zone FI — dispatch stack

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fi.index, y=fi['nuclear_gen_mw'],
    name='Nuclear', stackgroup='gen', line=dict(width=0),
    fillcolor='steelblue', mode='lines'))

fig.add_trace(go.Scatter(
    x=fi.index, y=fi['hydro_gen_mw'],
    name='Hydro', stackgroup='gen', line=dict(width=0),
    fillcolor='teal', mode='lines'))

if 'wind_mw' in fi.columns:
    fig.add_trace(go.Scatter(
        x=fi.index, y=fi['wind_mw'],
        name='Wind', stackgroup='gen', line=dict(width=0),
        fillcolor='lightgreen', mode='lines'))

# Elspot bid as a line overlay
fig.add_trace(go.Scatter(
    x=fi.index, y=fi['elspot_bid_mw'],
    name='Elspot bid', mode='lines',
    line=dict(color='black', dash='dot', width=1.5)))

fig.update_layout(title='Zone FI — dispatch stack (MW)',
                  xaxis_title='UTC', yaxis_title='MW',
                  hovermode='x unified', height=400)
fig.show()

## FCR-N commitment

In [ ]:
fcr_cols = [c for c in fi.columns if c.startswith('fcr_n')]

if fcr_cols and fi[fcr_cols].sum().sum() > 0:
    fig = go.Figure()
    colors = {'fcr_n_hydro_mw': 'teal', 'fcr_n_nuclear_mw': 'steelblue', 'fcr_n_total_mw': 'darkorange'}
    for col in fcr_cols:
        fig.add_trace(go.Scatter(
            x=fi.index, y=fi[col],
            name=col.replace('_', ' '), mode='lines',
            line=dict(color=colors.get(col, 'gray'))))
    fig.update_layout(title='FCR-N capacity committed (MW)',
                      xaxis_title='UTC', yaxis_title='MW',
                      hovermode='x unified', height=350)
    fig.show()
else:
    print('FCR-N not committed in this run (FCR-N may be disabled or prices too low).')

## Reservoir level

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fi.index, y=fi['reservoir_gwh'],
    name='FI reservoir', mode='lines', line=dict(color='teal', width=2),
    fill='tozeroy', fillcolor='rgba(0,128,128,0.1)'))

fig.update_layout(title='Hydro reservoir level (GWh)',
                  xaxis_title='UTC', yaxis_title='GWh',
                  hovermode='x unified', height=300)
fig.show()

## Zone SE2 — pump storage dispatch

In [ ]:
if se2 is not None:
    units = ['kymmen', 'letten', 'eggsjoen']
    colors_gen  = ['royalblue', 'seagreen', 'darkorange']
    colors_pump = ['lightblue', 'lightgreen', 'moccasin']

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=('Generation (MW)', 'Reservoir (GWh)'),
                        vertical_spacing=0.08)

    for unit, cg, cp in zip(units, colors_gen, colors_pump):
        gen_col  = f'{unit}_gen_mw'
        pump_col = f'{unit}_pump_mw'
        res_col  = f'{unit}_reservoir_gwh'

        if gen_col in se2.columns:
            fig.add_trace(go.Scatter(x=se2.index, y=se2[gen_col],
                                     name=f'{unit} gen', line=dict(color=cg)),
                          row=1, col=1)
        if pump_col in se2.columns:
            fig.add_trace(go.Scatter(x=se2.index, y=-se2[pump_col],
                                     name=f'{unit} pump (neg)', line=dict(color=cp, dash='dot')),
                          row=1, col=1)
        if res_col in se2.columns:
            fig.add_trace(go.Scatter(x=se2.index, y=se2[res_col],
                                     name=f'{unit} reservoir', line=dict(color=cg, dash='dot')),
                          row=2, col=1)

    fig.update_layout(title='Zone SE2 — pump storage (Kymmen / Letten / Eggsjön)',
                      hovermode='x unified', height=500)
    fig.show()
else:
    print('No SE2 data for this run.')

## Elspot bid curves — Day 1

In [ ]:
if bids_path.exists():
    bids = pd.read_csv(bids_path)
    print(f'Bid steps: {bids["step"].nunique()}  |  Hours: {bids["hour"].nunique()}')
    bids.head(6)
else:
    print(f'Bid file not found: {bids_path}')
    bids = None

In [ ]:
if bids is not None:
    # Plot bid curves for a selection of hours (e.g. hours 6, 12, 18)
    hours_to_plot = sorted(bids['hour'].unique())[:24:4]  # every 4th hour
    fig = go.Figure()

    palette = ['royalblue', 'seagreen', 'darkorange', 'crimson', 'purple', 'teal']
    for i, h in enumerate(hours_to_plot):
        sub = bids[bids['hour'] == h].sort_values('price_eur_mwh')
        fig.add_trace(go.Scatter(
            x=sub['quantity_mw'], y=sub['price_eur_mwh'],
            mode='lines+markers', name=f'Hour {h}',
            line=dict(color=palette[i % len(palette)])))

    fig.update_layout(title='Elspot bid curves — Day 1 (selected hours)',
                      xaxis_title='Quantity (MW)', yaxis_title='Price (EUR/MWh)',
                      hovermode='closest', height=400)
    fig.show()

## Revenue summary

In [ ]:
# Quick revenue estimate from schedule (requires a price series to be meaningful;
# this is approximate — the optimizer's objective is the ground truth).
summary = pd.DataFrame({
    'Metric': [
        'FI hydro avg (MW)', 'FI nuclear avg (MW)', 'FI elspot bid avg (MW)',
        'FI FCR-N total avg (MW)',
        'SE2 elspot bid avg (MW)' if se2 is not None else 'SE2',
        'Horizon (hours)',
    ],
    'Value': [
        fi['hydro_gen_mw'].mean(),
        fi['nuclear_gen_mw'].mean(),
        fi['elspot_bid_mw'].mean(),
        fi['fcr_n_total_mw'].mean() if 'fcr_n_total_mw' in fi.columns else 0,
        se2['elspot_bid_mw'].mean() if se2 is not None else float('nan'),
        len(fi),
    ],
}).set_index('Metric').round(1)
summary